# RAG Definition
Retrieval - Augmentation - Generation

LLMs are like a very smart student who have read the entire internet but have a "cutoff date". They don't know what happened yesterday, and they don't know what's inside your private files.

RAG (Retrieval-Augmented Generation) is the industry-standard way to fix this **without the massive cost of retraining the AI**. A RAG system retrieves relevant data to provide accurate responses.

| Features | RAG | Fine-Tuning |
|---|---|---|
|Updates| Instant: Just add a new file to the DB. | Slow: requires time to retraining |
|Accuracy| Higher: Can cite specific resources | High: Can still halucinate facts |
|Cost| Low: just use of-the-shelf LLMs Models | High: requires high power computing to train model |
|Privacy| Secure: Data stays in your database. | Risky: data is baked in to the model |

### Important Terms:
- Retrieval: finding the most relevant information.  
- Augmented: adding that information to the prompt.  
- Generation: creating the final answer from the model.  
- Knowledge Base: the stored documents/data to search in.  
- Chunks: small pieces of a document.  
- Embedding: a numeric vector that represents text meaning.

### Big idea of RAG
1. Ingest data (documents, plain text, images — use OCR on images if needed) into your knowledge base.  
2. Split documents into chunks that fit the model's context window.  
3. Convert chunks to embeddings with an embedding model and store them in a vector database.  
4. At query time, retrieve relevant chunks via nearest-neighbor search in the vector DB.  
5. Augment the prompt with those passages.  
6. Use the LLM to generate the final answer conditioned on the augmented context.

# Codebase
Hands-on minimal implementation: ingestion → chunking → embedding → storage → retrieval → generation

In [ ]:
# Configuration
# pip: numpy requests
import os
import hashlib
import requests
import numpy as np

# Set these or use environment variables
EMBEDDING_API_URL = os.getenv('EMBEDDING_API_URL')
EMBEDDING_API_KEY = os.getenv('EMBEDDING_API_KEY')
LLM_API_URL = os.getenv('LLM_API_URL')
LLM_API_KEY = os.getenv('LLM_API_KEY')

# Fallback embedding dim for deterministic local embeddings
EMBEDDING_DIM = 32

## Ingestion
Load documents into a list of dicts: {id, text, meta}.

In [ ]:
# Load example documents (replace with real loader for your files)
def load_docs():
    docs = [
        {
            'id': 'doc1',
            'text': 'RAG stands for Retrieval-Augmented Generation. It combines retrieval from a knowledge base with an LLM to produce grounded answers.',
            'meta': {'source': 'notes.md', 'page': 1}
        },
        {
            'id': 'doc2',
            'text': 'Chunking splits long documents into small passages. Overlap helps preserve context between chunks.',
            'meta': {'source': 'chunking.md', 'page': 2}
        }
    ]
    return docs

docs = load_docs()
len(docs)

In [ ]:
# Chunking: sliding-window (character-based) with overlap
def chunk_text(text, chunk_size=400, overlap=50):
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

# Quick test
chunk_text('A' * 900, chunk_size=300, overlap=50)[:2]

In [ ]:
# Deterministic fallback embedding using SHA256 digest -> fixed-length vector
def text_to_fallback_embedding(text, dim=EMBEDDING_DIM):
    h = hashlib.sha256(text.encode('utf-8')).digest()
    arr = np.frombuffer(h, dtype=np.uint8).astype(np.float32)
    if arr.size >= dim:
        vec = arr[:dim]
    else:
        vec = np.pad(arr, (0, dim - arr.size), mode='wrap')
    # normalize
    norm = np.linalg.norm(vec)
    return (vec / (norm + 1e-9)).tolist()

def get_embeddings(texts):
    # If you have a real embedding API, call it here (batching recommended).
    # Example request shape (generic): {'texts': [...]} with Authorization header.
    if EMBEDDING_API_URL and EMBEDDING_API_KEY:
        try:
            resp = requests.post(EMBEDDING_API_URL, json={'texts': texts},
                                 headers={'Authorization': f'Bearer {EMBEDDING_API_KEY}'} , timeout=15)
            resp.raise_for_status()
            payload = resp.json()
            # adapt this depending on provider response shape:
            return [item['embedding'] for item in payload['data']]
        except Exception as e:
            print('Embedding API error, falling back to local:', e)
    # Fallback (deterministic)
    return [text_to_fallback_embedding(t) for t in texts]

In [ ]:
# Simple in-memory vector store
class InMemoryVectorStore:
    def __init__(self):
        self.vectors = []  # list of dicts: {id, embedding (list), text, meta}

    def upsert(self, items):
        for it in items:
            self.vectors.append(it)

    def _cosine(self, a, b):
        a = np.array(a, dtype=np.float32)
        b = np.array(b, dtype=np.float32)
        denom = (np.linalg.norm(a) * np.linalg.norm(b)) + 1e-9
        return float(np.dot(a, b) / denom)

    def search_by_vector(self, query_vec, k=3):
        scores = []
        for v in self.vectors:
            score = self._cosine(query_vec, v['embedding'])
            scores.append({'id': v['id'], 'score': score, 'text': v['text'], 'meta': v.get('meta')})
        scores.sort(key=lambda x: x['score'], reverse=True)
        return scores[:k]

    def __len__(self):
        return len(self.vectors)

# create store
store = InMemoryVectorStore()

## Retrieval
Nearest-neighbor search against the vector store (cosine similarity).

In [ ]:
# Build prompt with retrieved contexts and simple citations
def build_prompt(query, contexts):
    header = f'Answer the question using only the provided sources. If the answer is not in the sources, say "I dont know."\n\nQuestion: {query}\n\nSources:\n'
    parts = []
    for i, c in enumerate(contexts, start=1):
        src = c.get('meta', {}).get('source', c.get('id'))
        parts.append(f"[Source {i}] {src} (score={c['score']:.3f}):\n{c['text']}\n")
    prompt = header + '\n---\n'.join(parts) + '\n\nAnswer:'
    return prompt

# Lightweight LLM call with safe fallback
def generate_with_llm(prompt, max_tokens=256, temperature=0.0):
    if LLM_API_URL and LLM_API_KEY:
        try:
            resp = requests.post(LLM_API_URL, json={'prompt': prompt, 'max_tokens': max_tokens, 'temperature': temperature},
                                 headers={'Authorization': f'Bearer {LLM_API_KEY}'} , timeout=30)
            resp.raise_for_status()
            out = resp.json()
            # adapt depending on provider
            return out.get('text') or out.get('output') or str(out)
        except Exception as e:
            print('LLM API error, falling back to local echo:', e)
    # Fallback: short extractive answer using the highest-score context
    return 'Based on the top source: ' + (prompt[:1000] if isinstance(prompt, str) else str(prompt))

## Augmentation
How retrieved contexts are added to the prompt and citation format.

In [ ]:
# Demo: end-to-end minimal pipeline
def build_index(docs, chunk_size=300, overlap=50):
    items = []
    for doc in docs:
        chunks = chunk_text(doc['text'], chunk_size=chunk_size, overlap=overlap)
        for i, ch in enumerate(chunks):
            item = {
                'id': f"{doc['id']}_c{i}",
                'text': ch,
                'meta': {**doc.get('meta', {}), 'parent_id': doc['id'], 'chunk_index': i},
            }
            items.append(item)
    # batch embed
    texts = [it['text'] for it in items]
    embeddings = get_embeddings(texts)
    for it, emb in zip(items, embeddings):
        it['embedding'] = emb
    return items

items = build_index(docs)
store.upsert(items)
print('Indexed vectors:', len(store))

# Query
query = 'What is RAG and why chunking matters?'
q_emb = get_embeddings([query])[0]
results = store.search_by_vector(q_emb, k=3)
print('\nRetrieved contexts:')
for r in results:
    print(f"- {r['id']} score={r['score']:.3f} meta={r['meta']}")

prompt = build_prompt(query, results)
print('\nPrompt preview:\n', prompt[:1000])
answer = generate_with_llm(prompt)
print('\nGenerated answer:\n', answer)

## Generation
Uses the LLM to answer based on augmented prompt. Keep temperature low for factual answers.

## Notes & Next steps
- **Config**: set `EMBEDDING_API_URL`, `EMBEDDING_API_KEY`, `LLM_API_URL`, `LLM_API_KEY` before using remote services.
- **Diagnostics**: print retrieved contexts and prompt to understand hallucinations.
- **Improvements**: add semantic chunking, richer metadata (page, author), caching, and batching for embedding calls.
- **Evaluation**: later add a small QA set and measure retrieval precision/recall and generation faithfulness.